<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/notebooks/7_Proteina_e_Agua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula Prática 07 - Dinâmica Molecular de uma Proteína em Água

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## 🎯 Objetivos da aula

Na Aula 06 simulamos água pura. Agora vamos simular uma **proteína em solução aquosa**, partindo de uma estrutura experimental. Ao final da aula você deverá ser capaz de:

1. Buscar uma proteína no **Protein Data Bank (PDB)**, avaliar a qualidade da estrutura e baixar o arquivo;
2. Preparar o sistema numa condição de solução **definida**: **pH** (estados de protonação) e **força iônica** (sal dissolvido);
3. Entender os truques que permitem um passo de tempo de **4 fs**;
4. Rodar a simulação (NVT → NPT) gravando a **trajetória** e os **dados termodinâmicos** no seu **Google Drive**, com *checkpoints* para retomar a simulação se o Colab desconectar;
5. Estimar o **custo computacional** de uma simulação de biomolécula.

> 📌 A **análise estrutural** da trajetória (RMSD, RMSF, raio de giração, estrutura secundária) será feita na **Aula 08**, a partir dos arquivos que você vai gravar hoje no Drive.

## 🗺️ Roteiro

| Parte | O que fazemos |
|---|---|
| 0 | Instalação, plataforma e conexão com o Google Drive |
| 1 | Buscando a proteína no PDB |
| 2 | O campo de força: Amber14 + TIP3P-FB |
| 3 | Preparando a solução: pH, solvente e força iônica |
| 4 | `System`, integrador e minimização |
| 5 | Simulação, arquivos de saída e *checkpoints* |
| 6 | Verificação rápida e arquivos no Drive |
| 7 | Custo computacional e hands-on |

## 🔗 O que muda em relação à Aula 06

| Aula 06 (água pura) | Aula 07 (proteína em água) |
|---|---|
| topologia montada à mão, átomo por átomo | topologia lida de um arquivo **PDB** experimental |
| moléculas colocadas numa rede cúbica | `Modeller.addSolvent` preenche a caixa com água pré-equilibrada |
| só o modelo de água (TIP3P) | campo de força de proteína (Amber14) + água (TIP3P-FB) + íons |
| sem química de solução | **pH** define as cargas da proteína; **força iônica** define o sal |
| $\Delta t = 2$ fs (água rígida) | $\Delta t = 4$ fs (vínculos nas ligações com H + massa repartida) |
| arquivos no disco temporário do Colab | arquivos no **Google Drive**, com *checkpoints* |

---
# Parte 0 — Preparando o ambiente

No Colab, ative a GPU em **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.

In [ ]:
%%capture
!pip install openmm[cuda12] mdtraj py3Dmol

In [ ]:
!python -m openmm.testInstallation

In [ ]:
import os
import re
import time
import urllib.request
from collections import Counter
from sys import stdout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import openmm as mm
import openmm.app as app
import openmm.unit as unit
import py3Dmol

In [ ]:
def escolhe_plataforma(preferencias=("CUDA", "OpenCL", "CPU")):
    for nome in preferencias:
        try:
            plataforma = mm.Platform.getPlatformByName(nome)
            teste = mm.System()
            teste.addParticle(1.0)
            mm.Context(teste, mm.VerletIntegrator(0.001), plataforma)
            print(f"Usando a plataforma {nome}")
            return plataforma
        except Exception:
            print(f"Plataforma {nome} indisponível")
    raise RuntimeError("Nenhuma plataforma disponível")

def pega_forca(system, tipo):
    return next(f for f in system.getForces() if isinstance(f, tipo))

plataforma = escolhe_plataforma()

## 0.1 Conectando o Google Drive

O disco do Colab é **temporário**: quando a sessão termina (ou cai por inatividade), todos os arquivos em `/content` são apagados. Uma simulação de 1 ns gera arquivos que vamos precisar na Aula 08, então vamos gravá-los diretamente no seu **Google Drive**.

Ao executar a célula abaixo, o Colab pede autorização para acessar o seu Drive. Depois disso, ele aparece como uma pasta comum em `/content/drive/MyDrive`, e tudo que for gravado ali aparece em [drive.google.com](https://drive.google.com).

> 💡 Se você estiver rodando o notebook fora do Colab (no seu computador), a célula simplesmente usa uma pasta local.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    pasta_base = "/content/drive/MyDrive"
    NO_COLAB = True
except ImportError:
    pasta_base = "."
    NO_COLAB = False

pasta = os.path.join(pasta_base, "EQE595", "Aula07")
os.makedirs(pasta, exist_ok=True)
print("Os arquivos desta aula serão gravados em:", pasta)

A partir de agora, todo arquivo será gravado com um caminho do tipo `os.path.join(pasta, "nome_do_arquivo")`. Uma função curta deixa isso mais legível:

In [ ]:
def caminho(nome):
    return os.path.join(pasta, nome)

print(caminho("teste.txt"))

---
# Parte 1 — Buscando a proteína no PDB

## 1.1 No site do RCSB PDB

O [RCSB Protein Data Bank](https://www.rcsb.org) é o repositório público de estruturas 3D de proteínas e ácidos nucleicos, com mais de 200 mil entradas. Cada entrada tem um **código de 4 caracteres**.

**Passo a passo:**

1. Acesse [www.rcsb.org](https://www.rcsb.org) e digite `lysozyme` na busca. Aparecem centenas de estruturas: de organismos diferentes, com mutações, com ligantes, obtidas por métodos diferentes...
2. Abra a página da entrada [**1AKI**](https://www.rcsb.org/structure/1AKI) (lisozima de clara de ovo de galinha). Localize:
   - o **método experimental** (*X-ray diffraction*, *Solution NMR*, *Electron microscopy*) e a **resolução**;
   - o **organismo** de origem;
   - as **macromoléculas** (quantas cadeias? quantos resíduos?);
   - os **ligantes** e moléculas pequenas presentes.
3. Em **Download Files → PDB Format** você baixa o arquivo `1AKI.pdb`. Note o endereço do link: `https://files.rcsb.org/download/1AKI.pdb`. É esse padrão que usaremos no código.

**Critérios para escolher uma estrutura para simular:**

| Critério | Por quê |
|---|---|
| boa resolução (de preferência < 2 Å) | posições atômicas mais confiáveis |
| estrutura **completa** (sem resíduos ou átomos faltando) | resíduos faltando precisam ser modelados antes de simular |
| proteína "selvagem" (sem mutações), do organismo de interesse | a simulação representa a proteína que nos interessa |
| ligantes conhecidos ou ausentes | ligantes exigem parâmetros próprios (como os da Aula sobre GAFF) |

## 1.2 Baixando o arquivo

Gravamos o arquivo original no Drive, para reutilizá-lo na Aula 08.

In [ ]:
codigo_pdb = "1AKI"

arquivo_pdb = caminho(f"{codigo_pdb}.pdb")
if not os.path.exists(arquivo_pdb):
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{codigo_pdb}.pdb", arquivo_pdb)
print(f"{arquivo_pdb}: {os.path.getsize(arquivo_pdb) / 1024:.0f} kB")

## 1.3 O que tem dentro de um arquivo PDB?

Um arquivo PDB é um arquivo de **texto**, organizado em registros de 80 colunas. Os primeiros 6 caracteres de cada linha dizem o tipo de registro. Vale a pena abrir o arquivo e olhar:

In [ ]:
with open(arquivo_pdb) as arq:
    linhas = arq.readlines()

registros = Counter(linha[:6].strip() for linha in linhas)
print("Registros mais frequentes:", dict(registros.most_common(12)))
print()
for tipo in ["HEADER", "EXPDTA", "SSBOND", "ATOM  ", "HETATM"]:
    print(next((l for l in linhas if l.startswith(tipo)), f"{tipo.strip()}: (não encontrado)").rstrip())

Os registros mais importantes:

| Registro | Conteúdo |
|---|---|
| `HEADER`, `TITLE`, `SOURCE` | classificação, título e organismo |
| `EXPDTA`, `REMARK 2` | método experimental e resolução |
| `REMARK 465`, `REMARK 470` | resíduos e átomos **faltando** na estrutura |
| `SEQRES` | sequência completa da proteína |
| `SSBOND` | pontes dissulfeto (ligações S–S entre cisteínas) |
| `ATOM` / `HETATM` | coordenadas dos átomos: nome, resíduo, cadeia, $x$, $y$, $z$ (Å), ocupação e **fator B** |

A função abaixo extrai um resumo do cabeçalho. Ela é uma checagem que deve ser feita **antes** de qualquer simulação:

In [ ]:
def resumo_pdb(linhas):
    info = {"título": "", "faltando: resíduos": 0, "faltando: átomos": 0}
    pontes, hetero = [], Counter()
    for l in linhas:
        reg = l[:6].strip()
        if reg == "HEADER":
            info["classificação"], info["data de depósito"] = l[10:50].strip(), l[50:59].strip()
        elif reg == "TITLE":
            info["título"] += l[10:80].strip() + " "
        elif reg == "EXPDTA":
            info["método"] = l[10:79].strip()
        elif reg == "SOURCE" and "ORGANISM_SCIENTIFIC" in l:
            info["organismo"] = l.split(":")[1].strip(" ;\n")
        elif l.startswith("REMARK   2 RESOLUTION"):
            info["resolução (Å)"] = float(re.findall(r"[\d.]+", l[22:])[0])
        elif re.match(r"REMARK 465 +[A-Z]{3} +[A-Z] +-?\d+", l):
            info["faltando: resíduos"] += 1
        elif re.match(r"REMARK 470 +[A-Z]{3} +[A-Z] +-?\d+", l):
            info["faltando: átomos"] += 1
        elif reg == "SSBOND":
            pontes.append(f"{l[11:14]}{l[17:21].strip()}–{l[25:28]}{l[31:35].strip()}")
        elif reg == "HETATM":
            hetero[l[17:20].strip()] += 1
    info["pontes dissulfeto"] = pontes
    info["HETATM (nº de átomos)"] = dict(hetero)
    return info

for chave, valor in resumo_pdb(linhas).items():
    print(f"{chave:22s}: {valor}")

> ❓ **Interprete:**
> - Qual o método e a resolução? A estrutura está completa?
> - Quantas pontes dissulfeto existem? Elas ajudam a explicar por que a lisozima é uma proteína tão estável.
> - Quais moléculas aparecem como `HETATM`? O que vamos fazer com elas?

## 1.4 Lendo a estrutura no OpenMM

O OpenMM lê o arquivo e cria a `Topology` e as posições, as mesmas peças que montamos à mão na Aula 06:

In [ ]:
pdb = app.PDBFile(arquivo_pdb)
print(pdb.topology)

contagem = Counter(res.name for res in pdb.topology.residues())
print("\nResíduos:", dict(sorted(contagem.items())))

n_H = sum(1 for atomo in pdb.topology.atoms() if atomo.element.symbol == "H")
print(f"\nÁtomos de hidrogênio no arquivo: {n_H}")

> ❓ Por que o arquivo **não tem nenhum hidrogênio**? (Dica: os raios X são espalhados pelos elétrons. Quantos elétrons tem um átomo de H?)

Visualizando a proteína (representação em *cartoon*, colorida do N-terminal em azul ao C-terminal em vermelho), as cisteínas em bastões e as águas do cristal em vermelho:

In [ ]:
vis = py3Dmol.view(width=600, height=450)
vis.addModel(open(arquivo_pdb).read(), "pdb")
vis.setStyle({"cartoon": {"color": "spectrum"}})
vis.addStyle({"resn": "HOH"}, {"sphere": {"radius": 0.3, "color": "red"}})
vis.addStyle({"resn": "CYS"}, {"stick": {}})
vis.zoomTo()
vis.show()

---
# Parte 2 — O campo de força

A forma funcional é **a mesma** da Aula 06:

$$
U = \sum_\text{lig.} \frac{k_l}{2}(l - l_0)^2 + \sum_\text{âng.} \frac{k_\theta}{2}(\theta-\theta_0)^2 + \sum_\text{diedros} k_\phi\left[1+\cos(n\phi-\phi_0)\right]
+ \sum_{i<j} 4\varepsilon_{ij}\left[\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} - \left(\frac{\sigma_{ij}}{r_{ij}}\right)^{6}\right] + \sum_{i<j}\frac{q_i q_j}{4\pi\varepsilon_0 r_{ij}}.
$$

O que muda é o **número de tipos de átomos e de parâmetros**: cada um dos 20 aminoácidos tem seu próprio conjunto de cargas parciais, e os **diedros** da cadeia principal determinam se a proteína forma hélices ou folhas.

Usaremos:
- `amber14-all.xml`: o campo de força **Amber ff14SB** para proteínas [Maier *et al.*, J. Chem. Theory Comput. **11**, 3696 (2015)], que também inclui parâmetros de íons;
- `amber14/tip3pfb.xml`: o modelo de água **TIP3P-FB** [Wang, Martínez e Pande, J. Phys. Chem. Lett. **5**, 1885 (2014)], uma reparametrização do TIP3P.

> ⚠️ Os campos de força de proteína são parametrizados **junto** com um modelo de água. Não se deve misturar arbitrariamente um campo de força de proteína com qualquer modelo de água.

In [ ]:
forcefield = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")

---
# Parte 3 — Preparando a solução: pH, solvente e força iônica

Toda a preparação é feita com o `Modeller`, uma "bancada de trabalho" que edita a topologia e as posições antes da simulação. As **condições da solução** ficam definidas nesta célula:

In [ ]:
pH           = 7.0                      # define os estados de protonação
forca_ionica = 0.15 * unit.molar        # NaCl adicionado (0,15 M ≈ soro fisiológico)
cation, anion = "Na+", "Cl-"            # outras opções: "K+", "Li+", "Cs+", "Rb+" / "F-", "Br-", "I-"
padding      = 1.0 * unit.nanometer     # distância mínima proteína–borda da caixa

## 3.1 Removendo as águas do cristal
As poucas águas resolvidas no cristal serão substituídas por uma caixa inteira de solvente.

In [ ]:
modeller = app.Modeller(pdb.topology, pdb.positions)
print("Átomos no arquivo PDB:        ", modeller.topology.getNumAtoms())
modeller.deleteWater()
print("Após remover as águas:        ", modeller.topology.getNumAtoms())

## 3.2 pH e estados de protonação

Os aminoácidos **ionizáveis** ganham ou perdem um próton conforme o pH da solução. Para um grupo ácido com constante $\text{p}K_a$, a equação de Henderson–Hasselbalch dá a razão entre as formas desprotonada e protonada:

$$\frac{[\text{A}^-]}{[\text{HA}]} = 10^{\,\text{pH} - \text{p}K_a}.$$

Assim, se $\text{pH} > \text{p}K_a$ predomina a forma desprotonada.

| Resíduo | Grupo | Forma em pH baixo | Forma em pH alto | p$K_a$ aprox. |
|---|---|---|---|---|
| Asp (ASP) | carboxila | ASH (0) | ASP (−1) | ~4 |
| Glu (GLU) | carboxila | GLH (0) | GLU (−1) | ~4,4 |
| His (HIS) | imidazol | HIP (+1) | HID/HIE (0) | ~6,5 |
| Cys (CYS) | tiol | CYS (0) | — | ~8,5 |
| Lys (LYS) | amina | LYS (+1) | LYN (0) | ~10 |
| Arg (ARG) | guanidina | ARG (+1) | ARG (+1) | ~12,5 |

O `addHydrogens` usa esses p$K_a$ **fixos** para escolher a forma de cada resíduo e depois coloca todos os hidrogênios em posições geometricamente razoáveis.

> ⚠️ Na proteína real, o p$K_a$ de um resíduo depende do seu ambiente (resíduos vizinhos, exposição ao solvente). Programas como PROPKA ou H++ estimam esses p$K_a$ deslocados. Para a maioria dos resíduos de superfície, os valores de referência são uma boa aproximação.

In [ ]:
modeller.addHydrogens(forcefield, pH=pH)
print("Após adicionar hidrogênios:   ", modeller.topology.getNumAtoms())

def forma_protonacao(res):
    # Identifica a forma do resíduo pelos hidrogênios presentes
    h = {a.name for a in res.atoms()}
    if res.name == "ASP": return "ASH (0)" if "HD2" in h else "ASP (−1)"
    if res.name == "GLU": return "GLH (0)" if "HE2" in h else "GLU (−1)"
    if res.name == "LYS": return "LYS (+1)" if "HZ3" in h else "LYN (0)"
    if res.name == "ARG": return "ARG (+1)"
    if res.name == "CYS": return "CYS (0)" if "HG" in h else "CYX (ponte S–S)"
    if res.name == "HIS":
        return {(True, True): "HIP (+1)", (True, False): "HID (0)", (False, True): "HIE (0)"}[("HD1" in h, "HE2" in h)]
    return None

formas = Counter(forma_protonacao(res) for res in modeller.topology.residues())
del formas[None]
print(f"\nFormas dos resíduos ionizáveis em pH {pH}:")
for forma, n in sorted(formas.items()):
    print(f"   {forma:16s} × {n}")

> 💡 **CYX** é a cisteína que forma ponte dissulfeto (sem H no enxofre). **HIE**/**HID** são as duas formas neutras da histidina (o H fica num ou noutro nitrogênio do anel); **HIP** é a forma protonada (+1).

### Carga líquida da proteína

A carga total é a soma das cargas parciais de todos os átomos. Calculamos pelo campo de força:

In [ ]:
def carga_liquida(topologia):
    system = forcefield.createSystem(topologia, nonbondedMethod=app.NoCutoff)
    f_nb = pega_forca(system, mm.NonbondedForce)
    return sum(f_nb.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge)
               for i in range(system.getNumParticles()))

print(f"Carga líquida da lisozima em pH {pH}: {carga_liquida(modeller.topology):+.2f} e")

cargas_formas = {"(+1)": +1, "(−1)": -1}
soma = sum(n * v for forma, n in formas.items() for k, v in cargas_formas.items() if k in forma)
print(f"Contagem pelas formas acima: {soma:+d} e  (os dois terminais, NH3+ e COO−, se cancelam)")

### Curva de titulação

Repetindo a preparação para vários valores de pH, obtemos a carga líquida da proteína em função do pH. O pH em que a carga se anula é o **ponto isoelétrico** (pI). Ele é a base de técnicas de **separação de proteínas** como a cromatografia de troca iônica e a focalização isoelétrica.

In [ ]:
def prepara_proteina(pH):
    m = app.Modeller(pdb.topology, pdb.positions)
    m.deleteWater()
    m.addHydrogens(forcefield, pH=pH)
    return m

valores_pH = np.arange(1.0, 14.1, 0.5)
cargas = [carga_liquida(prepara_proteina(p).topology) for p in valores_pH]

plt.figure(figsize=(7, 4))
plt.step(valores_pH, cargas, where="mid", lw=2)
plt.axhline(0, color="k", lw=0.8)
plt.axvline(pH, color="C3", ls="--", label=f"pH da simulação ({pH})")
plt.xlabel("pH"); plt.ylabel("Carga líquida (e)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

> ❓ **Discuta:**
> - A lisozima é carregada positivamente ou negativamente em pH 7? Quais resíduos são responsáveis por isso?
> - O pI experimental da lisozima é ≈ 11. O modelo de p$K_a$ fixos consegue anular a carga? Por que não? (Dica: veja a tabela de p$K_a$. A Arg é titulada pelo `addHydrogens`? E a Tyr?)
> - Em que faixa de pH você usaria uma resina de troca **catiônica** para reter a lisozima?

## 3.3 Solvente e força iônica

`addSolvent` cria uma caixa periódica, preenche com água pré-equilibrada (removendo as moléculas que se sobrepõem à proteína) e adiciona íons em duas etapas:

1. `neutralize=True` (padrão): adiciona **contra-íons** até a carga total ser zero. Isso é obrigatório com PME: a soma de Ewald de um sistema periódico carregado não é bem definida.
2. `ionicStrength`: adiciona pares cátion/ânion extras até atingir a força iônica pedida.

A **força iônica** é definida como
$$ I = \frac{1}{2}\sum_i c_i z_i^2, $$
e para um sal 1:1 como o NaCl ela é igual à concentração do sal. Ela controla a **blindagem** das interações eletrostáticas pelos íons. O alcance dessa blindagem é o **comprimento de Debye**, que na água a 25 °C vale
$$ \lambda_D \approx \frac{0{,}304\ \text{nm}}{\sqrt{I/\text{M}}}. $$

In [ ]:
modeller.addSolvent(forcefield, model="tip3p", padding=padding,
                    ionicStrength=forca_ionica, positiveIon=cation, negativeIon=anion, neutralize=True)

vetores = modeller.topology.getPeriodicBoxVectors()
L = vetores[0][0].value_in_unit(unit.nanometer)
print(f"Caixa: {L:.2f} × {L:.2f} × {L:.2f} nm")

composicao = Counter()
for res in modeller.topology.residues():
    tipo = "água" if res.name == "HOH" else ("íon" if len(res) == 1 else "proteína")
    composicao[tipo] += len(res)
n_total = modeller.topology.getNumAtoms()
for tipo, n_at in composicao.items():
    print(f"   {tipo:9s}: {n_at:6d} átomos ({100 * n_at / n_total:4.1f} %)")
print(f"   {'total':9s}: {n_total:6d} átomos")

ions = Counter(res.name for res in modeller.topology.residues() if len(res) == 1)
n_agua = sum(1 for res in modeller.topology.residues() if res.name == "HOH")
print(f"\nMoléculas de água: {n_agua}")
print(f"Íons: {dict(ions)}")
I_M = forca_ionica.value_in_unit(unit.molar)
if I_M > 0:
    print(f"Comprimento de Debye: {0.304 / np.sqrt(I_M):.2f} nm")

> ❓ **Confira as contas:**
> - Por que há mais ânions do que cátions? A diferença bate com a carga líquida da proteína?
> - O `Modeller` calcula o número de pares de íons por $n_\text{pares} = N_\text{água} \times I / (55{,}4\ \text{M})$, onde 55,4 M é a concentração molar da água pura. Verifique esse número com os dados acima.
> - O comprimento de Debye é maior ou menor que a distância entre a proteína e suas imagens periódicas?

Visualizando o sistema montado: proteína em *cartoon*, íons em esferas (Na⁺ em roxo, Cl⁻ em verde) e água em linhas finas.

In [ ]:
arquivo_inicial = caminho("lisozima_inicial.pdb")
with open(arquivo_inicial, "w") as arq:
    app.PDBFile.writeFile(modeller.topology, modeller.positions, arq, keepIds=True)

vis = py3Dmol.view(width=600, height=450)
vis.addModel(open(arquivo_inicial).read(), "pdb")
vis.setStyle({"resn": "HOH"}, {"line": {"opacity": 0.15}})
vis.addStyle({"not": {"resn": ["HOH", "NA", "CL"]}}, {"cartoon": {"color": "spectrum"}})
vis.addStyle({"resn": "NA"}, {"sphere": {"radius": 1.0, "color": "purple"}})
vis.addStyle({"resn": "CL"}, {"sphere": {"radius": 1.2, "color": "green"}})
vis.zoomTo()
vis.show()

> ❓ Que fração dos átomos é água? A maior parte do custo computacional é gasta com o **solvente**, e não com a proteína que nos interessa.

---
# Parte 4 — `System`, integrador e minimização

## 4.1 Por que 4 fs?

Na Aula 06 vimos que o passo de tempo é limitado pelo movimento mais rápido. Numa proteína:

| Movimento | Período típico | O que fazemos |
|---|---|---|
| estiramento X–H (C–H, N–H, O–H) | ~10 fs | `constraints=HBonds`: comprimentos fixos |
| ângulos envolvendo H | ~20 fs | `hydrogenMass=1.5 amu`: H mais pesados → oscilam mais devagar |
| demais ligações e ângulos | ≳ 20 fs | — |

A **repartição de massa dos hidrogênios** (*hydrogen mass repartitioning*, HMR) transfere massa do átomo pesado para o hidrogênio ligado a ele. A massa total de cada molécula (e portanto as propriedades de equilíbrio) não muda, mas as vibrações mais rápidas ficam mais lentas. Com HBonds + HMR, o `LangevinMiddleIntegrator` funciona bem com $\Delta t = 4$ fs.

In [ ]:
temperatura = 298.15 * unit.kelvin
pressao     = 1 * unit.bar
dt          = 4 * unit.femtosecond
gamma       = 1 / unit.picosecond

system = forcefield.createSystem(modeller.topology,
                                 nonbondedMethod=app.PME,
                                 nonbondedCutoff=1.0 * unit.nanometer,
                                 constraints=app.HBonds,
                                 hydrogenMass=1.5 * unit.amu)

atomos = list(modeller.topology.atoms())
H = next(a for a in atomos if a.element.symbol == "H")
N = next(a for a in atomos if a.element.symbol == "N")
print(f"Massa de {H.name} ({H.residue.name}{H.residue.id}): {system.getParticleMass(H.index)}  (H normal: 1.008 Da)")
print(f"Massa de {N.name} ({N.residue.name}{N.residue.id}): {system.getParticleMass(N.index)}  (N normal: 14.007 Da)")
print(f"Vínculos: {system.getNumConstraints()}")

## 4.2 Minimização

A estrutura cristalográfica, com hidrogênios recém-colocados e água inserida ao redor, tem pequenos choques entre átomos. Minimizamos a energia antes de iniciar a dinâmica.

In [ ]:
integrator = mm.LangevinMiddleIntegrator(temperatura, gamma, dt)
simulation = app.Simulation(modeller.topology, system, integrator, plataforma)
simulation.context.setPositions(modeller.positions)

def U_total(sim):
    return sim.context.getState(getEnergy=True).getPotentialEnergy().in_units_of(unit.kilojoule_per_mole)

print(f"Energia potencial antes da minimização:  {U_total(simulation)}")
simulation.minimizeEnergy()
print(f"Energia potencial depois da minimização: {U_total(simulation)}")

---
# Parte 5 — Simulação

## 5.1 Os arquivos de saída

Uma simulação produz vários arquivos, cada um com uma finalidade:

| Arquivo | Formato | Conteúdo | Para que serve |
|---|---|---|---|
| `lisozima_inicial.pdb` | texto | topologia + posições iniciais | define **quem é quem** na trajetória (o DCD só tem coordenadas) |
| `lisozima.dcd` | binário | posições de todos os átomos, a cada 2 ps | análise estrutural (Aula 08) |
| `lisozima_log.csv` | texto | $t$, $U$, $T$, $V$, $\rho$, desempenho, a cada 1 ps | análise termodinâmica |
| `lisozima.chk` | binário | estado completo (posições, velocidades, caixa, gerador aleatório) | **retomar** uma simulação interrompida |
| `system_npt.xml` | texto | o `System` completo (forças, parâmetros, barostato) | recriar a simulação sem refazer a preparação |
| `lisozima_final.pdb`, `estado_final.xml` | texto | configuração final | visualizar; continuar a simulação depois |

> 💡 Por que DCD e não PDB para a trajetória? O PDB é texto: cada coordenada ocupa 8 caracteres (8 bytes), enquanto no DCD ela ocupa 4 bytes, sem repetir nomes de átomos e resíduos a cada quadro. Com ~20 mil átomos e 500 quadros, a diferença chega a gigabytes.

Os tempos de simulação e os intervalos de gravação:

In [ ]:
t_nvt = 40 * unit.picosecond
t_npt = 1000 * unit.picosecond      # 1 ns

passos_nvt = int(round(t_nvt / dt))
passos_npt = int(round(t_npt / dt))
intervalo_log  = 250       # passos → a cada 1 ps
intervalo_traj = 500       # passos → a cada 2 ps
intervalo_chk  = 12500     # passos → a cada 50 ps
print(f"NVT: {passos_nvt} passos;  NPT: {passos_npt} passos")
print(f"Quadros na trajetória: {(passos_nvt + passos_npt) // intervalo_traj}")
print(f"Tamanho estimado do DCD: {(passos_nvt + passos_npt) // intervalo_traj * n_total * 3 * 4 / 1e6:.0f} MB")

## 5.2 *Reporters* gravando no Drive

Os *reporters* gravam diretamente na pasta do Drive. O `CheckpointReporter` sobrescreve o mesmo arquivo `.chk` periodicamente, guardando sempre o estado mais recente.

In [ ]:
simulation.reporters = []
simulation.reporters.append(app.DCDReporter(caminho("lisozima.dcd"), intervalo_traj))
simulation.reporters.append(app.StateDataReporter(
    caminho("lisozima_log.csv"), intervalo_log, step=True, time=True,
    potentialEnergy=True, totalEnergy=True, temperature=True, volume=True, density=True, speed=True))
simulation.reporters.append(app.CheckpointReporter(caminho("lisozima.chk"), intervalo_chk))
simulation.reporters.append(app.StateDataReporter(
    stdout, 25000, step=True, time=True, temperature=True, density=True,
    progress=True, remainingTime=True, speed=True, totalSteps=passos_nvt + passos_npt))

## 5.3 Equilibração NVT

Sorteamos as velocidades de Maxwell–Boltzmann e deixamos o sistema relaxar a volume fixo.

> 💡 Em trabalhos de pesquisa, costuma-se **restringir as posições** dos átomos pesados da proteína durante a equilibração, para que o solvente se acomode sem deformar a estrutura. Aqui omitimos essa etapa por simplicidade.

In [ ]:
simulation.context.setVelocitiesToTemperature(temperatura)
simulation.step(passos_nvt)

## 5.4 Simulação NPT

Ligamos o barostato de Monte Carlo (Aula 06) e **gravamos o `System` completo** (já com o barostato) em XML. Esse arquivo, junto com o PDB inicial e o *checkpoint*, permite retomar a simulação sem refazer a preparação. Também medimos o **tempo de parede** gasto, para estimar o custo na Parte 7.

In [ ]:
system.addForce(mm.MonteCarloBarostat(pressao, temperatura, 25))
simulation.context.reinitialize(preserveState=True)

with open(caminho("system_npt.xml"), "w") as arq:
    arq.write(mm.XmlSerializer.serialize(system))

inicio = time.perf_counter()
simulation.step(passos_npt)
tempo_parede = time.perf_counter() - inicio
print(f"Tempo de parede da etapa NPT: {tempo_parede/60:.1f} min")

Ao final, gravamos a configuração final em dois formatos: PDB (para visualizar) e XML de estado (posições, velocidades e caixa, portável entre plataformas, para continuar a simulação depois).

In [ ]:
simulation.saveCheckpoint(caminho("lisozima.chk"))
simulation.saveState(caminho("estado_final.xml"))

estado = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
modeller.topology.setPeriodicBoxVectors(estado.getPeriodicBoxVectors())
with open(caminho("lisozima_final.pdb"), "w") as arq:
    app.PDBFile.writeFile(modeller.topology, estado.getPositions(), arq, keepIds=True)

del simulation    # fecha os arquivos dos reporters

## 5.5 E se o Colab desconectar? 🔌

Sessões gratuitas do Colab caem por inatividade ou por tempo de uso. Como os arquivos estão no Drive, **não se perde nada além do trecho desde o último *checkpoint***. Para retomar:

1. Execute de novo a Parte 0 (instalação, importações, plataforma e Drive) e a célula de parâmetros da Parte 5.1 (`t_nvt`, `t_npt`, intervalos). Defina também `temperatura`, `dt` e `gamma` (Parte 4.1);
2. Mude `RETOMAR = True` na célula abaixo e execute-a.

A célula recria a simulação a partir dos arquivos do Drive (`lisozima_inicial.pdb`, `system_npt.xml`, `lisozima.chk`), descobre quantos passos já foram feitos e continua do ponto em que parou, **acrescentando** dados aos arquivos existentes (`append=True`).

> ⚠️ Os quadros gravados entre o último *checkpoint* e a queda aparecem duas vezes nos arquivos. Na análise, remova as linhas repetidas pelo número do passo (`df.drop_duplicates("#\"Step\"", keep="last")`). Além disso, o arquivo `.chk` só pode ser lido na mesma plataforma (por exemplo, a mesma GPU). Se ela mudar, use `estado_final.xml` com `simulation.loadState(...)`.
>
> Se a queda ocorrer ainda na etapa NVT (que é curta), é mais simples recomeçar do zero.

In [ ]:
RETOMAR = False

if RETOMAR:
    pdb_ini = app.PDBFile(caminho("lisozima_inicial.pdb"))
    with open(caminho("system_npt.xml")) as arq:
        system = mm.XmlSerializer.deserialize(arq.read())
    simulation = app.Simulation(pdb_ini.topology, system,
                                mm.LangevinMiddleIntegrator(temperatura, gamma, dt), plataforma)
    simulation.loadCheckpoint(caminho("lisozima.chk"))

    passos_feitos = simulation.currentStep
    passos_restantes = passos_nvt + passos_npt - passos_feitos
    print(f"Retomando no passo {passos_feitos} (t = {passos_feitos * dt}); faltam {passos_restantes} passos")

    simulation.reporters.append(app.DCDReporter(caminho("lisozima.dcd"), intervalo_traj, append=True))
    simulation.reporters.append(app.StateDataReporter(
        caminho("lisozima_log.csv"), intervalo_log, step=True, time=True,
        potentialEnergy=True, totalEnergy=True, temperature=True, volume=True, density=True,
        speed=True, append=True))
    simulation.reporters.append(app.CheckpointReporter(caminho("lisozima.chk"), intervalo_chk))
    simulation.reporters.append(app.StateDataReporter(
        stdout, 25000, step=True, time=True, temperature=True, progress=True,
        remainingTime=True, speed=True, totalSteps=passos_nvt + passos_npt))
    simulation.step(passos_restantes)

    simulation.saveCheckpoint(caminho("lisozima.chk"))
    simulation.saveState(caminho("estado_final.xml"))
    estado = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
    pdb_ini.topology.setPeriodicBoxVectors(estado.getPeriodicBoxVectors())
    with open(caminho("lisozima_final.pdb"), "w") as arq:
        app.PDBFile.writeFile(pdb_ini.topology, estado.getPositions(), arq, keepIds=True)
    del simulation

---
# Parte 6 — Verificação rápida e arquivos no Drive

## 6.1 A simulação está equilibrada?

Antes de encerrar, verificamos rapidamente as séries temporais gravadas no Drive (o procedimento é o da Aula 06). A análise completa, termodinâmica e estrutural, fica para a Aula 08.

In [ ]:
df = pd.read_csv(caminho("lisozima_log.csv"))
df = df.drop_duplicates(df.columns[0], keep="last")     # remove repetições, caso a simulação tenha sido retomada
t = df["Time (ps)"]
t_nvt_ps = t_nvt.value_in_unit(unit.picosecond)

grandezas = [("Potential Energy (kJ/mole)", r"$U$ (kJ/mol)", "C0"),
             ("Temperature (K)", r"$T$ (K)", "C3"),
             ("Density (g/mL)", r"$\rho$ (g/mL)", "C2")]
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, (coluna, rotulo, cor) in zip(axs, grandezas):
    ax.plot(t, df[coluna], color=cor, lw=0.7)
    ax.axvline(t_nvt_ps, color="k", ls="--", lw=1)
    ax.set_ylabel(rotulo)
axs[-1].set_xlabel("Tempo (ps)")
plt.tight_layout()

prod = df[t > 100]
print(f"<T> = {prod['Temperature (K)'].mean():.1f} K   (alvo: {temperatura})")
print(f"<ρ> = {1000 * prod['Density (g/mL)'].mean():.1f} kg/m³")

> ❓ A temperatura flutua em torno do valor alvo? A densidade se estabilizou depois que o barostato foi ligado? Ela é maior ou menor que a da água pura da Aula 06? Por quê? (Dica: qual é a densidade de uma proteína? E de uma solução de NaCl?)

## 6.2 O que ficou gravado no Drive

In [ ]:
for nome in sorted(os.listdir(pasta)):
    print(f"   {nome:25s} {os.path.getsize(caminho(nome)) / 1e6:9.2f} MB")

Confira em [drive.google.com](https://drive.google.com), na pasta **EQE595/Aula07**. Esses arquivos serão o ponto de partida da Aula 08.

---
# Parte 7 — Custo computacional

A métrica padrão de desempenho em dinâmica molecular é **ns/dia**: quantos nanossegundos de simulação se obtêm em 24 h de cálculo.

In [ ]:
ns_simulados = t_npt.value_in_unit(unit.nanosecond)
ns_por_dia = ns_simulados / (tempo_parede / 86400)
print(f"{n_total} átomos, Δt = {dt}")
print(f"Desempenho: {ns_por_dia:.0f} ns/dia  →  {tempo_parede/60/ns_simulados:.1f} min por ns")

Para referência, numa T4 do Colab foram medidos:

| Tempo de simulação | Tempo computacional |
|---|---|
| 0,4 ns | 1 min 50 s |
| 1,0 ns | 4 min 40 s |
| 10,0 ns | ~40 min |

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON 1: quanto custa? &#x1F528;
</span>**

> Com o desempenho medido acima, estime quanto tempo levaria simular 10 ns, 100 ns e 1 µs. Compare com as escalas de tempo dos processos biológicos: flutuações de cadeias laterais (ps–ns), movimentos de alças (ns–µs), enovelamento de proteínas pequenas (µs–ms) e de proteínas como a lisozima (ms–s).
>
> Quantos passos de integração há em 1 µs com $\Delta t = 4$ fs? E com $\Delta t = 1$ fs, sem vínculos e sem HMR?

In [ ]:
for t_alvo_ns in [10, 100, 1000]:
    horas = t_alvo_ns / ns_por_dia * 24
    print(f"{t_alvo_ns:6d} ns → {horas:8.1f} h  ({horas/24:6.1f} dias)")

print(f"\nPassos em 1 µs com 4 fs: {(1 * unit.microsecond) / (4 * unit.femtosecond):.1e}")
print(f"Passos em 1 µs com 1 fs: {(1 * unit.microsecond) / (1 * unit.femtosecond):.1e}")

---
# Hands-on

Os exercícios abaixo **não exigem simulação**: só a preparação do sistema, que leva segundos. A função a seguir reúne a Parte 3.

In [ ]:
def prepara_solucao(pH=7.0, forca_ionica=0.15*unit.molar, padding=1.0*unit.nanometer,
                    cation="Na+", anion="Cl-"):
    # Retorna o Modeller com a proteína protonada, solvatada e com íons
    m = app.Modeller(pdb.topology, pdb.positions)
    m.deleteWater()
    m.addHydrogens(forcefield, pH=pH)
    carga = carga_liquida(m.topology)
    m.addSolvent(forcefield, model="tip3p", padding=padding, ionicStrength=forca_ionica,
                 positiveIon=cation, negativeIon=anion)
    residuos = Counter(res.name for res in m.topology.residues())
    ions = {nome: n for nome, n in residuos.items() if nome not in ("HOH",) and len(nome) <= 2}
    return {"pH": pH, "I (M)": forca_ionica.value_in_unit(unit.molar),
            "padding (nm)": padding.value_in_unit(unit.nanometer), "carga proteína": round(carga),
            "águas": residuos["HOH"], **ions, "átomos": m.topology.getNumAtoms()}

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON 2: pH e força iônica &#x1F528;
</span>**

> Prepare o sistema para as combinações abaixo e monte uma tabela com a carga da proteína e o número de cada íon:
> - pH 4, 7 e 10, com $I = 0{,}15$ M;
> - $I = 0$, $0{,}15$ e $0{,}5$ M, em pH 7.
>
> Explique a variação do número de Cl⁻ com o pH. Calcule o comprimento de Debye em cada força iônica e discuta: em qual delas as cargas da superfície da proteína "enxergam" umas às outras a distâncias maiores?

In [ ]:
casos = [prepara_solucao(pH=p) for p in [4.0, 7.0, 10.0]]
casos += [prepara_solucao(forca_ionica=I * unit.molar) for I in [0.0, 0.5]]
tabela = pd.DataFrame(casos).fillna(0)
tabela["λ_D (nm)"] = [0.304 / np.sqrt(I) if I > 0 else np.inf for I in tabela["I (M)"]]
tabela

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON 3: tamanho da caixa &#x1F528;
</span>**

> Prepare o sistema com `padding` de 0,8, 1,0, 1,5 e 2,0 nm.
> - Como o número de átomos cresce com o `padding`? (Dica: a aresta da caixa é $L \approx d + 2\,\text{padding}$, onde $d$ é o tamanho da proteína.)
> - Supondo que o custo seja proporcional ao número de átomos, estime o desempenho (ns/dia) de cada caixa a partir do seu resultado da Parte 7.
> - Que critério físico limita o `padding` mínimo?

In [ ]:
casos = pd.DataFrame([prepara_solucao(padding=p * unit.nanometer) for p in [0.8, 1.0, 1.5, 2.0]])
casos["ns/dia (estimado)"] = ns_por_dia * n_total / casos["átomos"]
casos

---
## 📌 Resumo

- Estruturas experimentais vêm do **PDB**. Antes de simular, avalie o método, a resolução, os resíduos faltando e os ligantes.
- Estruturas de raios X **não têm hidrogênios**. O `addHydrogens(pH=...)` escolhe os estados de protonação a partir de p$K_a$ de referência, e isso define a **carga** da proteína.
- O `addSolvent` adiciona água, **contra-íons** (neutralização, obrigatória com PME) e sal até a **força iônica** desejada.
- Vínculos nas ligações com H + repartição de massa dos hidrogênios permitem $\Delta t = 4$ fs.
- Grave os resultados no **Google Drive**: trajetória (DCD), dados termodinâmicos (CSV), topologia (PDB), `System` (XML) e *checkpoints* (CHK), que permitem retomar uma simulação interrompida.
- Simulações atômicas alcançam ns–µs. Muitos processos biológicos ocorrem em escalas de tempo muito maiores.

### Para saber mais
- [RCSB Protein Data Bank](https://www.rcsb.org/) e o [guia do formato PDB](https://www.wwpdb.org/documentation/file-format)
- [OpenMM Cookbook](https://openmm.github.io/openmm-cookbook/)

In [ ]:
# Ao final da aula: garante que tudo foi gravado no Drive antes de encerrar a sessão
if NO_COLAB:
    drive.flush_and_unmount()